# DroneFuse - Step 1: extract MMAUD recordings, group `v1` (no GPU needed)

**Recordings:** V1 (rooftop): Avata, M300, Mavic2, Mavic3, Pham4 (77.9 GB of ROS bags on the Hugging Face Hub).

The bags are **streamed** from the official MMAUD release, never stored: each worker downloads ~64 MB of
consecutive chunks, decompresses them, and keeps a compact copy of every sensor:
- camera: left half of the stereo pair, native 1280x960, JPEG, 10 Hz (every 3rd frame)
- Livox Avia and Mid-360 LiDAR: non-zero points (x, y, z, reflectivity), 5 Hz (every 2nd scan)
- radar: `/radar_enhanced_pcl` (x, y, z + Doppler, Range, Power, Alpha, Beta) and `/radar_trk`, 15 Hz
- audio: the original MP3 frames of all 4 nodes (nothing lost)
- Leica ground truth (position of the drone), all messages

**Kaggle settings:** Accelerator = **None (CPU)** - no GPU quota used. Internet = **On**.
**Inputs to add:** your `dronefuse-code` dataset only.
**Safe for "Save & Run All":** every recording is one step; an error is recorded and the next recording runs;
`step1_summary.json` and `extract_report.txt` are always written. Extraction stops starting new work 15 min before
the time budget, so the notebook always finishes inside Kaggle's 12-hour limit.
**Expected run time:** 3-5 hours. Output: a few GB in `/kaggle/working/dronefuse_outputs/step1_v1`.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# indexed_bzip2 may decompress faster than Python's bz2 module; it is only used if a speed test
# below shows it is faster AND gives identical bytes. huggingface_hub lists the files on the Hub.
import importlib
import subprocess

for package, module in (('huggingface_hub', 'huggingface_hub'), ('indexed_bzip2', 'indexed_bzip2')):
    try:
        importlib.import_module(module)
    except ImportError:
        result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], capture_output=True, text=True)
        print(f'pip install {package}:', 'ok' if result.returncode == 0 else result.stderr[-500:])

# The MMAUD README asks for the plain HTTP download path instead of the Xet client.
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
from dronefuse.extract import ExtractSettings

GROUP = 'v1'                  # v1 | v2a | v2b  (one notebook per group)
ONLY_RECORDINGS = None              # e.g. ['V2_m3e'] to re-run just one recording of the group
REPO_ID = 'Yuan-Shenghai/MMAUD'
OUTPUT = Path('/kaggle/working/dronefuse_outputs/step1_' + GROUP)
TIME_BUDGET_MINUTES = 11 * 60 + 15  # Kaggle stops a session at 12 h
STOP_MARGIN_MINUTES = 15            # no new extraction work is started in the last 15 min
WORKERS = os.cpu_count() or 4       # one worker process per CPU
SETTINGS = ExtractSettings(
    camera_every=3,                 # 30 Hz -> 10 Hz
    lidar_every=2,                  # 10 Hz -> 5 Hz
    camera_half='left',
    camera_size=None,               # None = native 1280x960 (the drone is only a few pixels wide)
    jpeg_quality=92,
    chunks_per_task=32,             # ~64 MB per download request
)

In [ ]:
import json
import time

from dronefuse import environment
from dronefuse.recordings import BY_NAME, GROUPS
from dronefuse.runner import StepRunner

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
deadline = runner.start + (TIME_BUDGET_MINUTES - STOP_MARGIN_MINUTES) * 60
summary = {'group': GROUP, 'recordings': {}}


def save_summary():
    summary['steps'] = runner.steps
    with open(OUTPUT / 'step1_summary.json', 'w') as f:
        json.dump(summary, f, indent=1, default=str)


summary['environment'] = runner.run('measure the machine', environment.collect)
print(json.dumps(summary['environment'], indent=1, default=str))
save_summary()

In [ ]:
# Every file on the Hub (to find each bag and its byte parts).
import requests
from dronefuse import hf_inventory
from dronefuse.probe import bag_sources

session = requests.Session()
files = runner.run('list every MMAUD file on the Hub', hf_inventory.list_files, REPO_ID) or []
sources = bag_sources(files)
recordings = [BY_NAME[name] for name in (ONLY_RECORDINGS or GROUPS[GROUP])]
for r in recordings:
    print(r.name, r.drone_type, '| sensor bag', r.sensor_bag, 'found' if r.sensor_bag in sources else 'MISSING',
          '| truth bag', 'found' if r.truth_bag in sources else 'MISSING')
save_summary()

In [ ]:
# Pick the faster bz2 decoder (tested on real chunks; must give identical bytes).
from dronefuse.extract import choose_bz2_decoder
from dronefuse.probe import open_bag
from dronefuse.rosbag_index import read_index


def test_decoders():
    reader = open_bag(REPO_ID, sources[recordings[0].sensor_bag], session)
    return choose_bz2_decoder(reader, read_index(reader))


choice = runner.run('choose the bz2 decoder', test_decoders)
if choice:
    SETTINGS.bz2, summary['bz2_decoder'] = choice
print('bz2 decoder:', SETTINGS.bz2, summary.get('bz2_decoder'))
save_summary()

In [ ]:
# The extraction: one step per recording. Progress is printed every 5 %.
from dronefuse.extract import run_recording

for recording in recordings:
    free_gb = environment.disk_report(['/kaggle/working']).get('/kaggle/working', {}).get('free_gb')
    print(f'free space in /kaggle/working: {free_gb} GB')
    if free_gb is not None and free_gb < 3:
        print('Less than 3 GB left in /kaggle/working - not starting', recording.name)
        summary['recordings'][recording.name] = None
        continue
    if time.time() > deadline:
        print('Time budget used up - not starting', recording.name)
        summary['recordings'][recording.name] = None
        continue
    meta = runner.run(f'extract {recording.name}', run_recording, REPO_ID, recording, sources, OUTPUT,
                      SETTINGS, WORKERS, deadline, session)
    summary['recordings'][recording.name] = meta
    save_summary()

In [ ]:
# The report: printed here and saved as extract_report.txt.
from dronefuse.extract import format_extract_report

report = format_extract_report(summary['recordings'], summary.get('environment'), summary.get('bz2_decoder'))
failed = runner.failed()
report += '\n\nSTEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed))
saved = sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())
report += f'\nSAVED: {saved / 1e9:.2f} GB in {OUTPUT}'
(OUTPUT / 'extract_report.txt').write_text(report)
save_summary()
print(report)

## After the run

1. Send back `dronefuse_outputs/step1_v1/extract_report.txt` (or the printed report above).
2. Turn the output into a dataset for Step 2: open this notebook's saved version -> **Output** tab ->
   **New Dataset**, name it **`dronefuse-mmaud-v1`** (private).
3. If a recording shows `STOPPED BY TIME BUDGET` or `FAILED`, tell me: it can be re-run on its own by setting
   `ONLY_RECORDINGS = ['<name>']` in the Settings cell of a copy of this notebook.